# Human Action Recognition: transfer learning with MobileNetV2
Classify images into 15 human actions using the HAR dataset (Nagadia, Kaggle) and a MobileNetV2 (Sandler et al., 2018) backbone pretrained on ImageNet (Deng et al., 2009). Transfer learning reuses ImageNet features, which suits a 12,600-image dataset (Yosinski et al., 2014).

## 0. Setup
The first cell runs on Kaggle (clones the repo, finds the dataset) or locally, and silences TensorFlow log noise so the training output stays readable.

In [ ]:
import os, sys, glob, subprocess, warnings
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

if os.path.exists("/kaggle/input"):
    REPO = Path("/kaggle/working/Human_Action_Recognition-mlops")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/s-gasaro/Human_Action_Recognition-mlops", str(REPO)], check=True)
    csv = glob.glob("/kaggle/input/**/Training_set.csv", recursive=True)[0]
    IMAGE_DIR = Path(csv).parent / "train"
    ON_KAGGLE = True
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
    IMAGE_DIR = REPO / "data" / "train"
    ON_KAGGLE = False

sys.path.insert(0, str(REPO / "src"))
SPLITS_CSV = REPO / "data" / "splits.csv"
MODELS_DIR = REPO / "models"
print(REPO, IMAGE_DIR, ON_KAGGLE)

**Result:** the paths point to the repo, the training images and `data/splits.csv`; `ON_KAGGLE` decides where the model is saved at the end.

## 1. Data loading
Load the stratified 70/15/15 split built by `src/preprocessing.py` (seed 42), so every experiment uses the same train/val/test images. The test set is only used for the final evaluation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from PIL import Image
import preprocessing as pp
import model as hm

tf.get_logger().setLevel("ERROR")
df = pp.load_splits(SPLITS_CSV)
class_names = pp.get_class_names(df)
train_df, val_df, test_df = [df[df.split == s] for s in ("train", "val", "test")]
print(len(train_df), len(val_df), len(test_df), len(class_names))
print("GPUs:", tf.config.list_physical_devices("GPU"))

**Result:** 8,820 / 1,890 / 1,890 images for train / val / test over 15 classes. Check that a GPU is listed above before training.

## 2. Exploratory data analysis
### Class distribution
Check class balance, since imbalance would make accuracy misleading and call for class weights.

In [ ]:
df.groupby(["label", "split"]).size().unstack().plot(kind="bar", stacked=True, figsize=(12, 4))
plt.ylabel("images"); plt.tight_layout(); plt.show()

**Result:** the dataset is balanced: every class has 840 images, split evenly across train/val/test. No class weighting is needed.

### Sample image per class
Look at one image per class to see how varied the scenes are.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, c in zip(axes.ravel(), class_names):
    f = train_df[train_df.label == c].sample(1, random_state=0).filename.iloc[0]
    ax.imshow(Image.open(IMAGE_DIR / f)); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

**Result:** one training example per class, to judge variety in background, framing and pose before choosing the augmentation.

### Image size and aspect ratio
Measure the raw sizes (1,000 random training images) to decide how to resize to 224x224.

In [ ]:
sample = train_df.sample(1000, random_state=0)
sizes = np.array([Image.open(IMAGE_DIR / f).size for f in sample.filename])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(sizes[:, 0], bins=30, alpha=.6, label="width"); ax[0].hist(sizes[:, 1], bins=30, alpha=.6, label="height"); ax[0].legend()
ax[1].hist(sizes[:, 0] / sizes[:, 1], bins=30); ax[1].set_title("aspect ratio (w/h)")
plt.tight_layout(); plt.show()
print("median size (w, h):", np.median(sizes, axis=0), "| landscape share:", (sizes[:, 0] > sizes[:, 1]).mean())

**Result:** images are small (median about 275x183) and mostly landscape (about 80%, median aspect ratio 1.5). Resizing to a 224x224 square therefore squeezes the width, and the pipeline does not preserve the aspect ratio.

### Brightness per class
Check whether lighting differs by class, which a model could pick up as a shortcut (20 images per class, mean gray level 0-255).

In [ ]:
rows = []
for c in class_names:
    for f in train_df[train_df.label == c].sample(20, random_state=0).filename:
        rows.append((c, np.asarray(Image.open(IMAGE_DIR / f).convert("L").resize((64, 64))).mean()))
b = pd.DataFrame(rows, columns=["label", "brightness"])
b.boxplot(by="label", column="brightness", figsize=(12, 4), rot=60); plt.suptitle(""); plt.ylabel("brightness"); plt.tight_layout(); plt.show()
print(b.groupby("label").brightness.mean().round(0).sort_values().to_dict())

**Result:** class means range from about 100 (`fighting`) to about 156 (`calling`), but the boxes overlap heavily. Brightness varies within each class more than between classes, so it is unlikely to be a strong shortcut.

## 3. Preprocessing and augmentation
Images are resized to 224x224 and scaled to [-1, 1] with MobileNetV2 `preprocess_input`. Augmentation is applied to training data only and is deliberately light: horizontal flip, rotation up to 5% of a turn and zoom up to 5%. Strong augmentation lowered validation F1 in the first Kaggle run.

In [ ]:
aug = pp.augmentation()
f = train_df.filename.iloc[0]
img = tf.image.resize(tf.io.decode_image(tf.io.read_file(str(IMAGE_DIR / f)), channels=3, expand_animations=False), pp.IMG_SIZE)[None]
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
axes[0].imshow(img[0].numpy().astype("uint8")); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(aug(img, training=True)[0].numpy().astype("uint8"))
for ax in axes: ax.axis("off")
plt.show()

make = lambda d, **k: pp.make_dataset(d, class_names, IMAGE_DIR, **k)
val_ds, test_ds = make(val_df), make(test_df)
train_plain = make(train_df, training=True, augment=False)
train_aug = make(train_df, training=True, augment=True)

**Result:** the augmented copies stay recognisable and differ only slightly from the original. `train_plain` (shuffled, no augmentation) and `train_aug` (shuffled, augmented) are built for the experiments.

## 4. Experiments
All models are MobileNetV2 with global average pooling, dropout and a 15-way softmax. They are compared on the validation set; training uses EarlyStopping and ReduceLROnPlateau from `src/model.py`. The helper below evaluates each model, plots its loss and accuracy curves and prints the best epoch and the train-validation accuracy gap at that epoch (a large gap means overfitting).

In [ ]:
results, models, histories = {}, {}, {}

def log(name, m, hist):
    h = hist.history
    results[name], models[name], histories[name] = hm.evaluate(m, val_ds), m, h
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    for a, k in zip(ax, ("loss", "accuracy")):
        a.plot(h[k], label="train"); a.plot(h["val_" + k], label="val")
        a.set_title(f"{name}: {k}"); a.set_xlabel("epoch"); a.legend()
    plt.tight_layout(); plt.show()
    r, e = results[name], int(np.argmin(h["val_loss"]))
    print(f"{name}: val F1 {r['f1']:.3f}, val acc {r['accuracy']:.3f}, best val_loss at epoch {e + 1}/{len(h['loss'])}, "
          f"train-val acc gap there {h['accuracy'][e] - h['val_accuracy'][e]:+.3f}")

### Exp 1: frozen base (baseline)
Train only the new head on top of the frozen ImageNet features, with no augmentation and no dropout, to get a baseline.

In [ ]:
m1 = hm.build_model(len(class_names), dropout=0.0, lr=1e-3)
h1 = hm.train(m1, train_plain, val_ds, epochs=8)
log("exp1_frozen", m1, h1)

**Conclusion:** this is the reference for every later experiment (about 0.69 validation F1 in the first Kaggle run). Validation loss flattening while training accuracy keeps rising signals overfitting of the head.

### Exp 2: frozen base + light augmentation + dropout 0.3
Same frozen setup as exp 1 plus the light augmentation and dropout 0.3, to see whether regularisation reduces the overfitting.

In [ ]:
m2 = hm.build_model(len(class_names), dropout=0.3, lr=1e-3)
h2 = hm.train(m2, train_aug, val_ds, epochs=10)
log("exp2_light_aug_dropout", m2, h2)

**Conclusion:** compare its val F1 and train-validation gap with exp 1. A smaller gap with equal or higher F1 means regularisation helped; a similar or lower F1 means the frozen features, not overfitting, limit accuracy (the earlier, heavier augmentation gave 0.66 versus 0.69).

### Exp 3: fine-tune the top 50 layers from exp 1 (LR 1e-5)
Copy the exp 1 model, unfreeze the top 50 layers of MobileNetV2 and continue with a very low learning rate (1e-5) so the pretrained weights are adjusted without being destroyed. It uses the unaugmented data of exp 1, up to 15 epochs with early stopping.

In [ ]:
m3 = tf.keras.models.clone_model(m1)
m3.set_weights(m1.get_weights())
hm.unfreeze(m3, 50, lr=1e-5)
h3 = hm.train(m3, train_plain, val_ds, epochs=15)
log("exp3_finetune_lr1e-5", m3, h3)

**Conclusion:** the gain over exp 1 is the effect of adapting the top layers. If val F1 barely moves, the head had already extracted most of what the features offer; a growing train-validation gap would mean the unfrozen layers are starting to overfit.

### Exp 4: same fine-tune with LR 1e-4
Repeat exp 3 from exp 1 with a 10x higher learning rate (1e-4) to find out how sensitive fine-tuning is to the learning rate.

In [ ]:
m4 = tf.keras.models.clone_model(m1)
m4.set_weights(m1.get_weights())
hm.unfreeze(m4, 50, lr=1e-4)
h4 = hm.train(m4, train_plain, val_ds, epochs=15)
log("exp4_finetune_lr1e-4", m4, h4)

**Conclusion:** exp 3 and exp 4 differ only in the learning rate. A lower F1 or an unstable validation loss for 1e-4 means it is too aggressive for the pretrained weights (in the first run, a frozen head with augmentation scored 0.62 at 1e-4).

## 5. Comparison on the validation set
One row per experiment. The best model by macro F1 (which weighs all 15 classes equally) is chosen here, so the test set stays untouched until the final evaluation.

In [ ]:
table = pd.DataFrame(results).T[["accuracy", "loss", "precision", "recall", "f1"]]
display(table.round(4))
best_name = table["f1"].idxmax()
best = models[best_name]
print(f"Best: {best_name} (val F1 {table.f1.max():.3f}, {table.f1.max() - table.loc['exp1_frozen', 'f1']:+.3f} vs the exp 1 baseline)")

**Conclusion:** the printed line gives the selected model and its margin over the frozen baseline. A margin of a few thousandths is within run-to-run noise and should not be over-interpreted.

## 6. Best model on the test set
Evaluate the selected model once on the held-out test set, with the classification report, the confusion matrix and per-class F1 to see which actions it handles badly.

In [ ]:
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
test_metrics = hm.evaluate(best, test_ds)
print(test_metrics)
y_true = np.concatenate([y.numpy().argmax(1) for _, y in test_ds])
y_pred = best.predict(test_ds, verbose=0).argmax(1)
print(classification_report(y_true, y_pred, target_names=class_names))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap="Blues"); ax.set_xticks(range(len(class_names)), class_names, rotation=90); ax.set_yticks(range(len(class_names)), class_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.tight_layout(); plt.show()

**Conclusion:** test macro F1 should be close to the validation value; a large drop would mean the model selection overfitted the validation set (the first Kaggle run gave 0.68 on test). Bright cells off the diagonal are the confused class pairs.

### Per-class F1
Rank classes by F1 to find where the model is weakest.

In [ ]:
f1s = precision_recall_fscore_support(y_true, y_pred, labels=range(len(class_names)), zero_division=0)[2]
order = np.argsort(f1s)
plt.figure(figsize=(8, 5)); plt.barh([class_names[i] for i in order], f1s[order]); plt.xlabel("F1"); plt.tight_layout(); plt.show()
print("Lowest F1:", {class_names[i]: round(float(f1s[i]), 2) for i in order[:3]})

**Conclusion:** the three classes with the lowest F1 are printed above; in the first Kaggle run they were `listening_to_music`, `sitting` and `texting`, which may look alike in a still image. Collecting more varied examples of these classes through the upload/retrain flow is the most direct way to improve them.

### Misclassified test images
Look at errors directly: 12 random misclassified test images with true and predicted labels, and the most frequent confusions.

In [ ]:
wrong = np.where(y_true != y_pred)[0]
pick = np.random.default_rng(0).choice(wrong, min(12, len(wrong)), replace=False)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax in axes.ravel(): ax.axis("off")
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(Image.open(IMAGE_DIR / test_df.iloc[i].filename))
    ax.set_title(f"true: {class_names[y_true[i]]}\npred: {class_names[y_pred[i]]}", fontsize=9)
plt.tight_layout(); plt.show()
print(Counter((class_names[t], class_names[p]) for t, p in zip(y_true[wrong], y_pred[wrong])).most_common(5))

**Conclusion:** the printed pairs are the most frequent (true, predicted) confusions on the test set. Check in the grid whether errors concentrate on ambiguous scenes or small subjects; small images squeezed to 224x224 may make these harder.

## 7. Prediction demo
Use `predict()` from `src/prediction.py`, the same function the API calls, on 3 random test images.

In [ ]:
from prediction import predict

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (_, r) in zip(axes, test_df.sample(3, random_state=1).iterrows()):
    res = predict(IMAGE_DIR / r.filename, best, class_names)
    ax.imshow(Image.open(IMAGE_DIR / r.filename)); ax.axis("off")
    ax.set_title(f"true: {r.label}\npred: {res['label']} ({res['confidence']:.2f})")
plt.tight_layout(); plt.show()

**Conclusion:** each title shows the true label, the predicted label and the model's confidence; `predict()` also returns the top-3 classes, which the API exposes.

## 8. Save the best model
Save the model, class names and metadata (experiment name and test metrics) with `save_model` to `models/`, and also to `/kaggle/working/models` on Kaggle so they can be downloaded.

In [ ]:
meta = {"experiment": best_name, **test_metrics}
hm.save_model(best, class_names, meta, MODELS_DIR)
if ON_KAGGLE:
    hm.save_model(best, class_names, meta, "/kaggle/working/models")
print(os.listdir(MODELS_DIR))

**Conclusion:** `har_model.h5`, `class_names.json` and `metadata.json` are the files the FastAPI service loads and that retraining replaces when a new version is promoted.

## 9. Discussion
The first Kaggle run motivated this revision: heavier augmentation (0.66 validation F1) and a 1e-4 learning rate (0.62) did not beat the frozen baseline (0.69), and fine-tuning from the augmented model only matched it. The experiments above therefore use lighter augmentation, fine-tune from the baseline and compare 1e-4 with 1e-5. The table in section 5 shows whether this helped; differences of a few thousandths are within noise on 1,890 validation images.

A frozen MobileNetV2 transfers well to this dataset (Sandler et al., 2018; Yosinski et al., 2014) but is limited by the data: images are small, squeezed to a square, and some classes may look alike in a still frame (`sitting`, `texting`, `listening_to_music` were weakest). Likely improvements are more varied images of weak classes (via the retrain flow), aspect-preserving resizing, and a larger backbone.

## References
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., Chen, L.-C. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. CVPR 2018.
- Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K., Fei-Fei, L. (2009). ImageNet: A large-scale hierarchical image database. CVPR 2009.
- Yosinski, J., Clune, J., Bengio, Y., Lipson, H. (2014). How transferable are features in deep neural networks? NeurIPS 2014.
- Nagadia, M. Human Action Recognition (HAR) Dataset. Kaggle. https://www.kaggle.com/datasets/meetnagadia/human-action-recognition-har-dataset